In [ ]:
import pandas as pd
import requests
import zipfile
import io
import os
from datetime import datetime, timedelta

# ========== Config ==========

gkg_folder = r"C:\Users\......."    #set output path here
os.makedirs(gkg_folder, exist_ok=True)

# Date range (for test, you can set start_date = end_date for one day)
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 11, 30)

flood_keywords = [
    'FLOOD'  # add any others relevant to the US
]
drought_keywords = [
    'DROUGHT', ,'IRRIGATION'# add any others
]

textual_keywords = [kw.lower() for kw in (flood_keywords + drought_keywords)]

# ========== Helper functions ==========

#this function retreives the part of the LOCATION field that contains the name of the location

def extract_district_state(location_str):
    if pd.isna(location_str):
        return None, None
    for loc in str(location_str).split(';'):
        parts = loc.split('#')
        if len(parts) >= 4:
            country_code = parts[2].upper()
            if country_code == 'IN':    # change this to US
                return parts[1], parts[3]
    return None, None



#this function retreives the 6 parts of the TONE field that contain different types of values, details in the codebook
def parse_tone(tone_str):
    """
    Expect six comma-separated values in tone_str.
    Return tuple of floats or Nones if invalid.
    """
    try:
        parts = str(tone_str).split(',')
        if len(parts) == 6:
            return tuple(float(p) for p in parts)
    except Exception:
        pass
    return (None, None, None, None, None, None)

def download_gkg(date_str):
    url = f"http://data.gdeltproject.org/gkg/{date_str}.gkg.csv.zip"
    print("Downloading:", date_str, "->", url)
    r = requests.get(url)
    if r.status_code != 200:
        print("  Failed, status:", r.status_code)
        return None
    z = zipfile.ZipFile(io.BytesIO(r.content))
    for fname in z.namelist():
        if fname.lower().endswith('.csv'):
            z.extract(fname, gkg_folder)
            return os.path.join(gkg_folder, fname)
    print("  No CSV file found in zip")
    return None


# Extract the parts of the THEME field that matched the keywords

def extract_matched_themes(themes_str, flood_kw, drought_kw):
    if not isinstance(themes_str, str):
        return ""
    parts = themes_str.split(';')
    matched = []
    for p in parts:
        p_up = p.upper()
        for kw in flood_kw:
            if kw.upper().replace(' ', '_') in p_up:
                matched.append(p)
                break
        else:
            for kw in drought_kw:
                if kw.upper().replace(' ', '_') in p_up:
                    matched.append(p)
                    break
    return ';'.join(matched)


# this function below  (find_textual_mentions) does NOT work yet - it is meant to directly retrieve some artcle text from GDELT
# hence , we need to use the urls and do our own cleaning and scraping , if rate limits allow it
def find_textual_mentions(quotes_str, docid, keywords):
    hits = []
    if isinstance(quotes_str, str):
        lower = quotes_str.lower()
        for kw in keywords:
            if kw in lower:
                hits.append(kw)
    if isinstance(docid, str):
        low = docid.lower()
        for kw in keywords:
            if kw in low:
                hits.append(kw)
    hits = list(sorted(set(hits)))
    return ';'.join(hits)

# ========== Main Loop ==========

all_records = []

current_date = start_date
while current_date <= end_date:
    date_str = current_date.strftime("%Y%m%d")
    csv_path = download_gkg(date_str)
    if not csv_path:
        current_date += timedelta(days=1)
        continue

    try:
        df = pd.read_csv(csv_path, sep='\t', dtype=str, low_memory=False)

        flood_pattern = '|'.join(flood_keywords)
        drought_pattern = '|'.join(drought_keywords)
        df['IsFlood'] = df['THEMES'].str.contains(flood_pattern, case=False, na=False)
        df['IsDrought'] = df['THEMES'].str.contains(drought_pattern, case=False, na=False)

        df_filtered = df[df['IsFlood'] | df['IsDrought']].copy()

        df_filtered[['District', 'State']] = df_filtered['LOCATIONS'].apply(
            lambda x: pd.Series(extract_district_state(x))
        )
        df_filtered = df_filtered.dropna(subset=['District', 'State'])

        def assign_event_type(r):
            if r['IsFlood']:
                return 'Flood'
            elif r['IsDrought']:
                return 'Drought'
            else:
                return None
        df_filtered['EventType'] = df_filtered.apply(assign_event_type, axis=1)

        df_filtered[['Tone_Avg', 'Tone_Positive', 'Tone_Negative', 'Tone_Polarity', 'Tone_ActivityRef', 'Tone_SelfGroupRef']] = \
            df_filtered['TONE'].apply(lambda x: pd.Series(parse_tone(x)))

        df_filtered['MatchedTheme'] = df_filtered['THEMES'].apply(
            lambda x: extract_matched_themes(x, flood_keywords, drought_keywords)
        )

        quote_col = None
        for col in df_filtered.columns:
            if col.upper().startswith('QUOTATION'):
                quote_col = col
                break

        docid_col = 'DOCUMENTIDENTIFIER' if 'DOCUMENTIDENTIFIER' in df_filtered.columns else None

        df_filtered['TextualMentions'] = df_filtered.apply(
            lambda r: find_textual_mentions(
                r.get(quote_col, ""), 
                r.get(docid_col, ""), 
                textual_keywords
            ), axis=1
        )

        # Include source fields (present in v1): SOURCES, SOURCEURLS
        output_cols = [
            'DATE', 'District', 'State', 'EventType', 'MatchedTheme',
            'SOURCES', 'SOURCEURLS',
            quote_col,
            'TextualMentions',
            'Tone_Avg', 'Tone_Positive', 'Tone_Negative', 'Tone_Polarity', 'Tone_ActivityRef', 'Tone_SelfGroupRef'
        ]

        output_cols = [c for c in output_cols if c in df_filtered.columns]

        sub = df_filtered[output_cols].copy()
        all_records.append(sub)

        print(date_str, "→", len(sub), "records matched and processed")

    except Exception as e:
        print("Error on", date_str, ":", e)

    try:
        os.remove(csv_path)
    except:
        pass

    current_date += timedelta(days=1)

if all_records:
    result_df = pd.concat(all_records, ignore_index=True)
    out_name = f"india_flood_drought_with_textual_jan_nov_2024.csv"
    out_path = os.path.join(gkg_folder, out_name)
    result_df.to_csv(out_path, index=False)
    print("✅ Saved result to:", out_path)
else:
    print("❌ No matching records found in the date range")
